In [7]:
import pandas as pd 
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score


from fantasy_football.fpl_api.get_performance_data import get_most_recent_gw_points
from fantasy_football.model.training.utils import load_gw_data, create_correlation_matrix, add_historic_rolling_features
from fantasy_football.utils import create_optimal_team
from fantasy_football.fpl_api.get_live_data import LivePlayerData

In [8]:
historic_data = load_gw_data("2025-26")
historic_data.head()

,name,position,team,xP,assists,bonus,bps,clean_sheets,creativity,element,...,transfers_in,transfers_out,value,was_home,yellow_cards,clearances_blocks_interceptions,defensive_contribution,recoveries,tackles,GW
0,Reinildo Mandava,DEF,Sunderland,0.5,0,0,27,1,2.5,541,...,0,0,40,True,0,6,8,3,2,1
1,Lewis Dobbin,MID,Aston Villa,1.0,0,0,0,0,0.0,57,...,0,0,50,True,0,0,0,0,0,1
2,Ryan Christie,MID,Bournemouth,0.0,0,0,0,0,0.0,87,...,0,0,50,False,0,0,0,0,0,1
3,Zeki Amdouni,FWD,Burnley,0.0,0,0,0,0,0.0,216,...,0,0,50,False,0,0,0,0,0,1
4,Lucas Tolentino Coelho de Lima,MID,West Ham,2.6,0,0,11,0,14.2,612,...,0,0,60,False,0,0,6,5,1,1


In [9]:
target_variable = "total_points"

historic_data[target_variable].describe()

count    29757.000000
mean         1.156333
std          2.352553
min         -3.000000
25%          0.000000
50%          0.000000
75%          1.000000
max         24.000000
Name: total_points, dtype: float64

# Known ahead of time

In [10]:
known_features = [
    "name",
    "element",
    "position",
    "team",
    "GW",
    "fixture",
    "was_home",
]

# Features from lagged columns

In [11]:
historic_features = [
    'assists',
    'clean_sheets',
    'creativity',
    'goals_conceded',
    'goals_scored',
    'ict_index',
    'influence',
    'own_goals',
    'penalties_missed',
    'penalties_saved',
    'red_cards',
    'saves',
    'selected',
    'starts',
    'threat',
    'transfers_balance',
    'value',
    'yellow_cards',
    'clearances_blocks_interceptions',
    'defensive_contribution',
    'recoveries',
    'tackles',
]

for feature in historic_features:

    historic_data = add_historic_rolling_features(
        historic_data,
        feature_column=feature,
        windows = [1] #, 3, 6, 9, 12)
)

calculated_features = [col for col in historic_data.columns if "calc_" in col]


In [12]:
correlation_matrix = create_correlation_matrix(
    historic_data, 
    columns=calculated_features, 
    target_variable=target_variable)

# Filter to include where correlation to target is >0.1
correlation_matrix = correlation_matrix[ (0.1 < correlation_matrix) &  (correlation_matrix < 0.9) ]
correlation_matrix

calc_starts_mean_last_1_gw                             0.506949
calc_ict_index_mean_last_1_gw                          0.444393
calc_defensive_contribution_mean_last_1_gw             0.436339
calc_influence_mean_last_1_gw                          0.423308
calc_recoveries_mean_last_1_gw                         0.421038
calc_clearances_blocks_interceptions_mean_last_1_gw    0.358353
calc_goals_conceded_mean_last_1_gw                     0.358055
calc_creativity_mean_last_1_gw                         0.339697
calc_tackles_mean_last_1_gw                            0.328383
calc_threat_mean_last_1_gw                             0.322386
calc_selected_mean_last_1_gw                           0.298265
calc_value_mean_last_1_gw                              0.288221
calc_clean_sheets_mean_last_1_gw                       0.242051
calc_goals_scored_mean_last_1_gw                       0.173602
calc_yellow_cards_mean_last_1_gw                       0.163167
calc_assists_mean_last_1_gw             

# MVP model

In [13]:
# Split by complete gameweeks so future results never leak into training
validation_fraction = 0.2
gameweeks = sorted(historic_data["GW"].dropna().unique())
split_index = max(1, int(len(gameweeks) * (1 - validation_fraction)))
validation_gameweeks = gameweeks[split_index:]

model_features = known_features + correlation_matrix.index.tolist()

train_mask = ~historic_data["GW"].isin(validation_gameweeks)
valid_mask = historic_data["GW"].isin(validation_gameweeks)

X_train = historic_data.loc[train_mask, model_features].copy()
y_train = historic_data.loc[train_mask, target_variable].copy()
X_valid = historic_data.loc[valid_mask, model_features].copy()
y_valid = historic_data.loc[valid_mask, target_variable].copy()

print(f"Train: {len(X_train):,} rows through GW {gameweeks[split_index - 1]}")
print(f"Validation: {len(X_valid):,} rows from GW {validation_gameweeks[0]}")


Train: 23,175 rows through GW 30
Validation: 6,582 rows from GW 31


In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

categorical_columns = ["name", "element", "position", "team", "fixture", "was_home"]
for frame in (X_train, X_valid):
    frame[categorical_columns] = frame[categorical_columns].fillna("__MISSING__").astype(str)

model = CatBoostRegressor(
    iterations=163,
    learning_rate=0.03,
    depth=6,
    loss_function="RMSE",
    random_seed=42,
    verbose=100,
)
model.fit(
    X_train,
    y_train,
    cat_features=categorical_columns,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=100,
)

predictions = model.predict(X_valid)
print(f"Validation MAE:  {mean_absolute_error(y_valid, predictions):.4f}")
print(f"Validation RMSE: {root_mean_squared_error(y_valid, predictions):.4f}")
print(f"Validation R²:   {r2_score(y_valid, predictions):.4f}")


0:	learn: 2.3532438	test: 2.2547937	best: 2.2547937 (0)	total: 61.1ms	remaining: 9.89s
100:	learn: 1.9322390	test: 1.8563490	best: 1.8563490 (100)	total: 248ms	remaining: 152ms
162:	learn: 1.9129260	test: 1.8550886	best: 1.8545172 (136)	total: 362ms	remaining: 0us

bestTest = 1.854517184
bestIteration = 136

Shrink model to first 137 iterations.
Validation MAE:  0.9435
Validation RMSE: 1.8545
Validation R²:   0.3349


In [15]:
validation_by_percentile = pd.DataFrame({
    "actual": y_valid.to_numpy(),
    "predicted": predictions,
})
validation_by_percentile["prediction_percentile"] = (
    pd.qcut(
        validation_by_percentile["predicted"],
        q=10,
        labels=False,
        duplicates="drop",
    ) + 1
)
validation_by_percentile["error"] = (
    validation_by_percentile["predicted"] - validation_by_percentile["actual"]
)
validation_by_percentile["absolute_error"] = validation_by_percentile["error"].abs()
validation_by_percentile["squared_error"] = validation_by_percentile["error"] ** 2

percentile_accuracy = validation_by_percentile.groupby("prediction_percentile").agg(
    observations=("actual", "size"),
    predicted_min=("predicted", "min"),
    predicted_max=("predicted", "max"),
    average_prediction=("predicted", "mean"),
    average_actual=("actual", "mean"),
    bias=("error", "mean"),
    mae=("absolute_error", "mean"),
    mean_squared_error=("squared_error", "mean"),
    within_one_point=("absolute_error", lambda errors: errors.le(1).mean()),
)
percentile_accuracy["rmse"] = percentile_accuracy["mean_squared_error"] ** 0.5
percentile_accuracy["within_one_point"] *= 100
percentile_accuracy.drop(columns="mean_squared_error").round(3)


,observations,predicted_min,predicted_max,average_prediction,average_actual,bias,mae,within_one_point,rmse
prediction_percentile,,,,,,,,,
1,671,0.013,0.039,0.025,0.010,0.014,0.035,99.851,0.116
2,652,0.039,0.042,0.041,0.000,0.041,0.041,100.000,0.041
3,653,0.042,0.066,0.050,0.043,0.007,0.090,99.541,0.430
4,659,0.066,0.110,0.089,0.056,0.033,0.139,98.634,0.330
5,657,0.110,0.268,0.171,0.131,0.040,0.278,97.412,0.637
6,657,0.268,0.788,0.507,0.557,-0.050,0.780,91.476,1.403
7,658,0.790,1.764,1.229,1.149,0.081,1.254,56.079,2.033
8,658,1.764,2.689,2.280,2.305,-0.026,1.917,30.547,2.737
9,658,2.689,3.315,3.008,2.950,0.058,2.253,22.340,2.978


In [16]:
from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_valid)

print(
    "Baseline MAE:",
    mean_absolute_error(y_valid, baseline_predictions),
)
print(
    "Baseline RMSE:",
    root_mean_squared_error(y_valid, baseline_predictions),
)
print(
    "Baseline R²:",
    r2_score(y_valid, baseline_predictions),
)

Baseline MAE: 1.4953238687971544
Baseline RMSE: 2.2762794411173592
Baseline R²: -0.0019516656118194753


# For initial draft, maximise the average forecast points for the last 8 weeks of last season

In [18]:
X_valid['prediction'] = model.predict(X_valid)

forecast_data = X_valid.groupby('name')['prediction'].mean().reset_index()

live_data = LivePlayerData()

# get ["name", "position", "team", "value"]
forecast_data['player_id'] = forecast_data['name'].map(live_data.get_player_id)
forecast_data['position'] = forecast_data['name'].map(live_data.get_live_player_position)
forecast_data['team'] = forecast_data['name'].map(live_data.get_live_player_team)
forecast_data['value'] = forecast_data['name'].map(live_data.get_live_player_cost)



In [19]:
forecast_data = forecast_data.sort_values(by="prediction", ascending=False).dropna()

In [20]:
optimal_team = create_optimal_team(forecast_data,'value')

optimal_team['last_gw_points'] = optimal_team['player_id'].map(get_most_recent_gw_points)

In [21]:
optimal_team

,name,prediction,player_id,position,team,value,last_gw_points
0,Erling Haaland,4.326364,411,FWD,Man City,155.0,2
1,Virgil van Dijk,3.979279,356,DEF,Liverpool,65.0,2
2,Harry Wilson,3.587049,260,MID,Leeds,65.0,3
3,James Tarkowski,3.564186,229,DEF,Everton,60.0,6
4,Nathan Collins,3.381042,84,DEF,Brentford,55.0,6
5,João Pedro Junqueira de Jesus,3.372321,165,FWD,Chelsea,76.0,11
6,Nikola Milenković,3.356914,471,DEF,Nott'm Forest,55.0,2
7,Bukayo Saka,3.234929,12,MID,Arsenal,95.0,9
8,Ola Aina,2.034922,473,DEF,Nott'm Forest,45.0,5
9,Gabriel Martinelli Silva,1.978059,18,MID,Arsenal,64.0,0


In [22]:
optimal_team.sum()

name              Erling HaalandVirgil van DijkHarry WilsonJames...
prediction                                                34.646375
player_id                411356260229841654711247318350166340297227
position              FWDDEFMIDDEFDEFFWDDEFMIDDEFMIDGKPFWDMIDMIDGKP
team              Man CityLiverpoolLeedsEvertonBrentfordChelseaN...
value                                                        1000.0
last_gw_points                                                   48
dtype: object

# Fiddly variables to come back to

In [23]:
#     "opponent_team",  # TODO: map to opponent team name from each season 
# game time of day
# Game day of week
# Last season performance


# Double game weeks
# What variable do i optimise for?